In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import torch
import einops as eo
import os
import rasterio as rio
from rasterio.features import rasterize

from c2s.models.chicken import ChickenConfig, ChickenOutput, Chicken

In [2]:
# Data pipeline parameters

# Data directory - should contain a subdirectory for each site
data_dir = 'data'

# File to put CSV data into
csv_file = 'dataset_file.csv'

# Classes can be commented out here if we wish to exclude them
class_map = {
    'lichen': 1,
    'rock': 4,
    'broadleaf': 5,
    'needleleaf': 6,
    'deadwood': 7,
    'graminoids': 8,
    'moss': 9,
    'soil': 10,
    'low_green': 12,
    'dry_branches': 13,
}

In [3]:
sites = os.listdir('data')
#sites = ['CS3A'] # override with a single site for testing purposes

In [4]:
def load_site_data(site: str):
    """Retrieves all the rgb / canopy height / label data for a drone site"""
    
    rgb_file = os.path.join(data_dir, site, f'{site}_hp_transparent_mosaic_group1.tif')
    chm_file = os.path.join(data_dir, site, f'{site}_hp_chm_stratified.tif')
    label_file = os.path.join(data_dir, site, f'{site}_hp_labelledpoints.gpkg')

    # load RGB raster and metadata
    with rio.open(rgb_file) as f:
        # Get raster metadata
        height = f.height
        width = f.width
        crs = f.crs
        transform = f.transform
        rgb_raster = f.read((1,2,3))
    
    # load chm file
    with rio.open(rgb_file) as f:
        chm_raster = f.read(1)
    
    # labels are stored as shapefiles, so we will need to rasterize it
    label_gdf = gpd.read_file(label_file).to_crs(crs)
    label_raster = rasterize(
        ((r['geometry'], r['class']) for _, r in label_gdf.iterrows()),
        out_shape=(height, width),
        transform=transform,
        dtype='uint8'
    )
    return rgb_raster, chm_raster, label_raster

In [5]:
def extract_pixel_features(rgb, chm, labels):
    # Get a list of indices where we have labelled pixels
    # Ex: [ [348, 1451], [581, 2099], ... ]
    indices = np.argwhere(np.isin(labels, list(class_map.values())))
    
    rgb_features = rgb[:,indices[:,0],indices[:,1]].T
    rgb_df = pd.DataFrame(rgb_features, columns=['R', 'G', 'B'])
    
    chm_features = chm[indices[:,0],indices[:,1]].reshape(-1, 1)
    chm_df = pd.DataFrame(chm_features, columns=['chm'])
    
    labels_array = labels[indices[:,0],indices[:,1]].reshape(-1, 1)
    labels_df = pd.DataFrame(labels_array, columns=['veg_class'])
    
    df = pd.concat([rgb_df, chm_df, labels_df], axis=1)
    return df, indices

In [6]:
def get_chicken_feature_map(rgb_raster):
    """ Use sliding-window DinoV2 to extract visual features
    rgb_raster: torch tensor with shape (B, C, H, W)
    returns:    torch tensor with shape (B, num_features, H, W)
    """
    rgb_torch_raster = torch.from_numpy(rgb_raster.astype(np.float32)).unsqueeze(0)
    b, c, h, w = rgb_torch_raster.shape
    new_h = int(h/14)*14
    new_w = int(w/14)*14
    rgb_cropped = rgb_torch_raster[:,:,:new_h,:new_w]
    chicken_config = ChickenConfig(
        vit_size="small",
        slider_buffer_size=1
    )
    chicken = Chicken(chicken_config)
    op = chicken._chicken_slider_infer(rgb_cropped).detach().numpy()
    return op

In [7]:
def extract_chicken_pixel_features(chicken_feature_raster, indices):
    # Find the relevant feature vector for labeled pixels. Since texture featuremap is downsampled,
    # we need to modify the indices we use.
    chicken_indices = (indices/14).astype(int)
    print(chicken_indices)
    chicken_pixel_features = chicken_feature_raster[0][:,chicken_indices[:,0],chicken_indices[:,1]].T
    print(chicken_pixel_features.shape)
    chicken_df = pd.DataFrame(chicken_pixel_features, columns=[f'chicken_{i}' for i in range(chicken_pixel_features.shape[1])])
    return chicken_df

In [8]:
all_features = None

for site in sites:
    print(f'Processing {site}...')
    rgb, chm, labels = load_site_data(sites[0])
    df1, indices = extract_pixel_features(rgb, chm, labels)
    df1['site'] = site

    # texture features
    chicken_feature_map = get_chicken_feature_map(rgb)
    df2 = extract_chicken_pixel_features(chicken_feature_map, indices)

    df = pd.concat([df1,df2], axis=1)

    if all_features is None:
        all_features = df
    else:
        all_features = pd.concat([all_features, df], axis=0)

    all_features.to_csv(csv_file, index=False)
    

Processing F3-20C...


Using cache found in /home/matt/.cache/torch/hub/facebookresearch_dinov2_main
xFormers not available
xFormers not available


[[ 21 266]
 [ 21 266]
 [ 21 266]
 ...
 [701 362]
 [701 362]
 [701 362]]
(1159, 384)
Processing CS117B...


Using cache found in /home/matt/.cache/torch/hub/facebookresearch_dinov2_main


[[ 21 266]
 [ 21 266]
 [ 21 266]
 ...
 [701 362]
 [701 362]
 [701 362]]
(1159, 384)
Processing CS3A...


...

[[ 21 266]
 [ 21 266]
 [ 21 266]
 ...
 [701 362]
 [701 362]
 [701 362]]
(1159, 384)
Processing CS3B...


Using cache found in /home/matt/.cache/torch/hub/facebookresearch_dinov2_main


[[ 21 266]
 [ 21 266]
 [ 21 266]
 ...
 [701 362]
 [701 362]
 [701 362]]
(1159, 384)
Processing F3-19C...


Using cache found in /home/matt/.cache/torch/hub/facebookresearch_dinov2_main


[[ 21 266]
 [ 21 266]
 [ 21 266]
 ...
 [701 362]
 [701 362]
 [701 362]]
(1159, 384)
